# RQ1 Effectiveness

How effective is the directed search in finding boundary inputs of VLMs compared with equal-budget baselines? This notebook reproduces the crossing-share table, the paired tests against the directed arm, the primary comparison with the random arm, the mixed model and the secondary comparisons with the image-only and DeepJanus-style arms. Run 00_collect first.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from mmb.analysis import stats as st, paired as pr, export as ex

RESULTS = Path("../results") if Path("../results").exists() else Path("results")
FIG = RESULTS / "figures"; FIG.mkdir(parents=True, exist_ok=True)
GRID = ["qwen", "internvl", "ministral"]      # the balanced grid; LLaVA-NeXT is the continuity SUT
REF = "a0_directed"                            # the directed arm, reference of every paired test
N_BOOT = 2000                                  # the paper uses 10000; raise for the final numbers
df = st.load_results([RESULTS / "MAIN.csv"])
scopes = ex.scopes(df, GRID)                   # {(sut, task): frame}; ("All", "All") pools the grid
grid_df = scopes[("All", "All")]
print(len(df), "runs;", "arms:", sorted(df["arm"].unique()))

## Crossing share per arm

Share of runs with a strict sign change of the pair contrast against the anchor, with a Wilson interval, the share already crossing in the initial population, the median evaluations to cross (censored at the budget) and the median floor. Pooled over the grid tasks and SUTs.

In [ ]:
summary = st.arm_summary(grid_df, N_BOOT)
summary

## Paired tests against the directed arm

Each arm against the directed arm on matched (cell, seed) pairs: exact McNemar for crossing, Wilcoxon signed-rank for evaluations and floor, rank-biserial effect sizes, Holm correction across arms within each metric.

In [ ]:
paired = st.paired_vs_reference(grid_df, REF) if REF in set(grid_df["arm"]) else None
paired if paired is not None else print("no directed-arm runs yet")

## Primary comparison

The pre-committed primary outcome: crossing share of the random arm against the directed arm per (task, SUT) stratum, intervals from a cell-cluster bootstrap, Holm across strata.

In [ ]:
primary = pr.primary_table(df, REF, "a1_random_tiered", N_BOOT)
primary

A mixed logit with random effects for cell and for SUT-task gives the odds ratio of the random arm relative to the directed arm. The per-cell figure shows whether a few cells drive the pooled difference.

In [ ]:
if REF in set(grid_df["arm"]):
    print(pr.mixed_model(grid_df.assign(cell_id=grid_df["cluster_id"]), REF, "a1_random_tiered"))
    pr.plot_per_cell_diff(df, FIG / "per-cell-diff.pdf", REF, "a1_random_tiered")
else:
    print("no directed-arm runs yet")

## Secondary comparisons

Image-only arm (are boundaries reachable without changing the question?) and DeepJanus-style arm against the directed arm, on matched pairs. These were declared after the primary plan and are not part of it.

In [ ]:
for arm in ("a7_image_only", "a9_deepjanus"):
    if arm in set(df["arm"]):
        display(pr.primary_table(df, REF, arm, N_BOOT))
    else:
        print(arm, "not run yet")